In [9]:
import numpy as np
import nashpy as nash

def calculadora_nash(A, B, nombres_filas=None, nombres_columnas=None):
    """
    Calcula e interpreta todos los equilibrios de Nash (puros y mixtos).
    A: Matriz de pagos Jugador 1 (np.array)
    B: Matriz de pagos Jugador 2 (np.array)
    """
    A = np.array(A, dtype=float)
    B = np.array(B, dtype=float)
    n_filas, n_cols = A.shape
    
    # Nombres por defecto si no se especifican
    if nombres_filas is None:
        nombres_filas = [f"F{i+1}" for i in range(n_filas)]
    if nombres_columnas is None:
        nombres_columnas = [f"C{j+1}" for j in range(n_cols)]
        
    juego = nash.Game(A, B)
    equilibrios = list(juego.support_enumeration())
    
    print("=" * 45)
    print("         RESULTADOS DEL JUEGO")
    print("=" * 45)
    
    if not equilibrios:
        print("❌ No se encontraron equilibrios.")
        return

    puros = []
    mixtos = []
    
    for sigma_r, sigma_c in equilibrios:
        # Es pura si las probabilidades son ceros y un uno exacto
        es_pura_r = np.any(np.isclose(sigma_r, 1.0))
        es_pura_c = np.any(np.isclose(sigma_c, 1.0))
        
        if es_pura_r and es_pura_c:
            idx_r = int(np.argmax(sigma_r))
            idx_c = int(np.argmax(sigma_c))
            pago_1 = A[idx_r, idx_c]
            pago_2 = B[idx_r, idx_c]
            puros.append((nombres_filas[idx_r], nombres_columnas[idx_c], pago_1, pago_2))
        else:
            pago_esp_1 = sigma_r @ A @ sigma_c
            pago_esp_2 = sigma_r @ B @ sigma_c
            mixtos.append((sigma_r, sigma_c, pago_esp_1, pago_esp_2))
            
    # Mostrar Estrategias Puras
    if puros:
        print(f"✅ Se encontraron {len(puros)} Equilibrio(s) de Nash en Estrategias PURAS:")
        for r, c, u1, u2 in puros:
            print(f"   👉 Estrategia: ({r}, {c})  |  Pagos: ({u1:g}, {u2:g})")
    else:
        print("⚠️  No hay equilibrios de Nash en estrategias puras.")
        
    # Mostrar Estrategias Mixtas
    if mixtos:
        print(f"\n🎲 Se encontraron {len(mixtos)} Equilibrio(s) en Estrategias MIXTAS:")
        for s_r, s_c, eu1, eu2 in mixtos:
            p1_str = ", ".join([f"{p:.2f} en {n}" for p, n in zip(s_r, nombres_filas) if p > 0.001])
            p2_str = ", ".join([f"{p:.2f} en {n}" for p, n in zip(s_c, nombres_columnas) if p > 0.001])
            print(f"   👉 Jugador 1 juega: [{p1_str}]")
            print(f"      Jugador 2 juega: [{p2_str}]")
            print(f"      Pagos esperados: ({eu1:.2f}, {eu2:.2f})\n")

In [10]:
# Ejemplo: Tu matriz 3x3
A_ejemplo = [
    [0, 4, 5],
    [4, 0, 5],
    [3, 3, 6]
]

B_ejemplo = [
    [4, 0, 3],
    [0, 4, 3],
    [5, 5, 6]
]

# Si quieres, puedes ponerle nombres a las estrategias (opcional)
calculadora_nash(A_ejemplo, B_ejemplo, 
                 nombres_filas=["Arriba", "Medio", "Abajo"], 
                 nombres_columnas=["Izq", "Centro", "Der"])

         RESULTADOS DEL JUEGO
✅ Se encontraron 1 Equilibrio(s) de Nash en Estrategias PURAS:
   👉 Estrategia: (Abajo, Der)  |  Pagos: (6, 6)


In [11]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import nashpy as nash

# 1. Controles para definir dimensiones
filas_w = widgets.Dropdown(options=[2, 3, 4], value=2, description='Filas:')
cols_w = widgets.Dropdown(options=[2, 3, 4], value=2, description='Columnas:')
boton_generar = widgets.Button(description='Regenerar Matriz', button_style='info')
boton_calcular = widgets.Button(description='Calcular Equilibrios', button_style='success')

matriz_box = widgets.VBox()
salida = widgets.Output()

entradas_A = []
entradas_B = []

def armar_grilla(b=None):
    global entradas_A, entradas_B
    entradas_A, entradas_B = [], []
    filas, cols = filas_w.value, cols_w.value
    filas_layout = []
    
    # Encabezado de columnas
    encabezado = [widgets.Label(value="", layout=widgets.Layout(width='60px'))]
    for j in range(cols):
        encabezado.append(widgets.Label(value=f"Col {j+1}", layout=widgets.Layout(width='125px', text_align='center')))
    filas_layout.append(widgets.HBox(encabezado))
    
    # Filas con casillas
    for i in range(filas):
        fila_widgets = [widgets.Label(value=f"Fila {i+1}:", layout=widgets.Layout(width='60px'))]
        fila_A, fila_B = [], []
        for j in range(cols):
            w_a = widgets.FloatText(value=0.0, layout=widgets.Layout(width='50px'))
            w_b = widgets.FloatText(value=0.0, layout=widgets.Layout(width='50px'))
            fila_A.append(w_a)
            fila_B.append(w_b)
            par = widgets.HBox([widgets.Label("("), w_a, widgets.Label(","), w_b, widgets.Label(")")])
            fila_widgets.append(par)
        entradas_A.append(fila_A)
        entradas_B.append(fila_B)
        filas_layout.append(widgets.HBox(fila_widgets))
        
    matriz_box.children = filas_layout

def resolver_juego(b):
    with salida:
        clear_output()
        if not entradas_A or not entradas_B:
            print("⚠️ Primero genera la matriz.")
            return
            
        A = np.array([[w.value for w in fila] for fila in entradas_A], dtype=float)
        B = np.array([[w.value for w in fila] for fila in entradas_B], dtype=float)
        
        juego = nash.Game(A, B)
        equilibrios = list(juego.support_enumeration())
        
        print("=" * 45)
        print("          RESULTADOS DEL JUEGO")
        print("=" * 45)
        
        if not equilibrios:
            print("❌ No se encontraron equilibrios.")
            return
            
        puros, mixtos = [], []
        for s_r, s_c in equilibrios:
            if np.all(np.isin(s_r, [0, 1])) and np.all(np.isin(s_c, [0, 1])):
                f, c = int(np.argmax(s_r)), int(np.argmax(s_c))
                puros.append((f + 1, c + 1, A[f, c], B[f, c]))
            else:
                pago_1 = s_r @ A @ s_c
                pago_2 = s_r @ B @ s_c
                mixtos.append((s_r, s_c, pago_1, pago_2))
                
        if puros:
            print(f"✅ {len(puros)} Equilibrio(s) en Estrategias PURAS:")
            for f, c, u1, u2 in puros:
                print(f"   👉 Estrategia: (Fila {f}, Columna {c}) | Pagos: ({u1:g}, {u2:g})")
        else:
            print("⚠️  No hay equilibrios puros.")
            
        if mixtos:
            print(f"\n🎲 {len(mixtos)} Equilibrio(s) en Estrategias MIXTAS:")
            for s_r, s_c, eu1, eu2 in mixtos:
                print(f"   👉 Prob. Jugador 1: {np.round(s_r, 2)}")
                print(f"      Prob. Jugador 2: {np.round(s_c, 2)}")
                print(f"      Pagos esperados: ({eu1:.2f}, {eu2:.2f})")

boton_generar.on_click(armar_grilla)
boton_calcular.on_click(resolver_juego)

# Generar grilla 2x2 inicial automáticamente
armar_grilla()

display(widgets.HBox([filas_w, cols_w, boton_generar]), matriz_box, boton_calcular, salida)

Button(button_style='success', description='Calcular Equilibrios', style=ButtonStyle())

Output()

In [12]:
import streamlit as st
import numpy as np
import nashpy as nash
import pandas as pd

st.set_page_config(page_title="Calculadora de Nash", page_icon="🎮", layout="centered")

st.title("🎮 Calculadora de Teoría de Juegos: Equilibrios de Nash")
st.markdown("Ingresa los pagos de cada jugador en la matriz para encontrar los equilibrios de Nash en estrategias puras y mixtas.")

# Configuración de dimensiones
col_dim1, col_dim2 = st.columns(2)
with col_dim1:
    filas = st.number_input("Número de Filas (Estrategias Jugador 1)", min_value=2, max_value=6, value=3, step=1)
with col_dim2:
    cols = st.number_input("Número de Columnas (Estrategias Jugador 2)", min_value=2, max_value=6, value=3, step=1)

st.subheader("📊 Matriz de Pagos (Forma Normal)")
st.caption("Escribe los pagos de cada casilla como: **pago1, pago2** (por ejemplo: `0, 4` o `6, 6`).")

# Valores por defecto para matriz 3x3 de ejemplo
default_vals = [
    ["0, 4", "4, 0", "5, 3"],
    ["4, 0", "0, 4", "5, 3"],
    ["3, 5", "3, 5", "6, 6"]
]

# Crear grilla de casillas
grid_inputs = []
for i in range(filas):
    cols_ui = st.columns(cols)
    row_vals = []
    for j in range(cols):
        with cols_ui[j]:
            def_val = default_vals[i][j] if i < 3 and j < 3 else "0, 0"
            val = st.text_input(f"F{i+1}, C{j+1}", value=def_val, key=f"cell_{i}_{j}")
            row_vals.append(val)
    grid_inputs.append(row_vals)

if st.button("🚀 Calcular Equilibrios de Nash", type="primary", use_container_width=True):
    try:
        # Parsear las matrices
        A = np.zeros((filas, cols), dtype=float)
        B = np.zeros((filas, cols), dtype=float)
        
        for i in range(filas):
            for j in range(cols):
                raw = grid_inputs[i][j].strip()
                partes = [p.strip() for p in raw.split(",") if p.strip()]
                if len(partes) != 2:
                    st.error(f"Error en F{i+1}, C{j+1}: ingresa dos números separados por coma (ejemplo: 5, 3).")
                    st.stop()
                A[i, j] = float(partes[0])
                B[i, j] = float(partes[1])
        
        juego = nash.Game(A, B)
        equilibrios = list(juego.support_enumeration())
        
        st.divider()
        st.subheader("🎯 Resultados del Análisis")
        
        if not equilibrios:
            st.warning("No se encontraron equilibrios con el algoritmo estándar.")
        else:
            puros = []
            mixtos = []
            
            for s_r, s_c in equilibrios:
                es_pura_r = np.any(np.isclose(s_r, 1.0))
                es_pura_c = np.any(np.isclose(s_c, 1.0))
                
                if es_pura_r and es_pura_c:
                    f = int(np.argmax(s_r))
                    c = int(np.argmax(s_c))
                    puros.append((f + 1, c + 1, A[f, c], B[f, c]))
                else:
                    eu1 = s_r @ A @ s_c
                    eu2 = s_r @ B @ s_c
                    mixtos.append((s_r, s_c, eu1, eu2))
            
            # Mostrar equilibrios puros
            if puros:
                st.success(f"**Se encontraron {len(puros)} Equilibrio(s) de Nash en Estrategias PURAS:**")
                for f, c, u1, u2 in puros:
                    st.markdown(f"- 👉 **(Fila {f}, Columna {c})** con pagos **({u1:g}, {u2:g})**")
            else:
                st.info("ℹ️ No existen equilibrios de Nash en estrategias puras.")
                
            # Mostrar equilibrios mixtos
            if mixtos:
                st.markdown("### 🎲 Equilibrios en Estrategias MIXTAS:")
                for idx, (s_r, s_c, eu1, eu2) in enumerate(mixtos, 1):
                    with st.expander(f"Equilibrio Mixto #{idx}", expanded=True):
                        col_m1, col_m2 = st.columns(2)
                        with col_m1:
                            st.write("**Probabilidades Jugador 1:**")
                            df_j1 = pd.DataFrame({"Estrategia": [f"Fila {k+1}" for k in range(filas)], "Probabilidad": s_r})
                            st.dataframe(df_j1, hide_index=True)
                            st.write(f"**Pago esperado J1:** {eu1:.2f}")
                        with col_m2:
                            st.write("**Probabilidades Jugador 2:**")
                            df_j2 = pd.DataFrame({"Estrategia": [f"Columna {k+1}" for k in range(cols)], "Probabilidad": s_c})
                            st.dataframe(df_j2, hide_index=True)
                            st.write(f"**Pago esperado J2:** {eu2:.2f}")
                            
    except Exception as e:
        st.error(f"Ocurrió un error en el cálculo: {e}")

2026-08-26 03:18:49.745 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-26 03:18:49.746 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-26 03:18:50.087 
  command:

    streamlit run C:\Users\santi\AppData\Roaming\Python\Python314\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-08-26 03:18:50.089 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-26 03:18:50.091 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-26 03:18:50.092 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-26 03:18:50.093 Thread 'MainThread': missing ScriptRunContext! This warning can b